# 22DM015 Final Project — Financial PhraseBank
## Part 3d — methodology analysis

This notebook is the student-authored Part 3d write-up.‍ It reads the shared scoreboard `results/results.csv` (produced by `part1.ipynb`, `bert_part2.ipynb`, `zeroshot_part2.ipynb` and `part3.ipynb`) and expresses every Part 2 technique as a point on the Part 3 data-scaling curve, so the methods can be compared on one common scale.‍ No model is trained here.‍

In [1]:
# watermark: AGLLM (AI-assisted content disclosure)
# Part 3d: read the shared scoreboard and express each Part 2 technique as a point on the
# Part 3 data-scaling curve - i.e. "how many real labels is each technique worth?".
# No training here: this only reads results/results.csv (run part1 / bert_part2 /
# zeroshot_part2 / part3 first so the rows exist).
import sys
import numpy as np
import pandas as pd
sys.path.append('..')
import eval_utils as eu

MODEL = 'bert-base-uncased'
res = pd.read_csv(eu.RESULTS_CSV)
res = res[res['split'] == 'test']

# The data-scaling curve = ONE technique (plain BERT, no augmentation), protocol 64/16/3.
curve = (res[(res['model'] == MODEL) & res['method'].str.startswith('full-')]
         .sort_values('n_train_labeled'))


def real_label_equivalent(f1):
    """Number of REAL training labels the curve needs to reach macro-F1 `f1`, by linear
    interpolation on the curve's (n_train_labeled, f1_macro) points. Returns np.inf when
    `f1` exceeds the curve's maximum (the full real dataset never matches it)."""
    xs = curve['n_train_labeled'].to_numpy(float)
    ys = curve['f1_macro'].to_numpy(float)        # monotonically increasing on this curve
    return np.inf if f1 > ys.max() else float(np.interp(f1, ys, xs))


# Part 2 techniques (protocol 128/8/20) + the zero-shot LLM, as comparison points.
TECHNIQUES = [('2c zero-shot (LLM)',   'claude-opus-4-8',   'zero-shot'),
              ('2d LLM-generated',     'bert-base-uncased', 'llm-generated'),
              ('2e optimal-combo',     'bert-base-uncased', 'optimal-combo'),
              ('2b back-translation',  'bert-base-uncased', 'augmented'),
              ('2a 32-shot',           'bert-base-uncased', '32-shot')]
rows = []
for label, model, method in TECHNIQUES:
    hit = res[(res['model'] == model) & (res['method'] == method)]
    if not len(hit):
        print(f'[waiting] no logged row yet for: {label}')
        continue
    r = hit.iloc[-1]
    eq = real_label_equivalent(float(r['f1_macro']))
    rows.append({'technique': label,
                 'rows_used': int(r['n_train_labeled']),
                 'f1_macro': round(float(r['f1_macro']), 4),
                 'real_label_equivalent': ('>full data' if np.isinf(eq) else round(eq))})

print('Data-scaling curve (plain BERT, protocol max_len 64 / batch 16 / 3 epochs):')
print(curve[['method', 'n_train_labeled', 'f1_macro']].to_string(index=False), '\n')
print('Part 2 techniques placed on the curve (protocol max_len 128 / batch 8 / 20 epochs):')
summary = pd.DataFrame(rows)
summary

Data-scaling curve (plain BERT, protocol max_len 64 / batch 16 / 3 epochs):
   method  n_train_labeled  f1_macro
  full-1%               15  0.283901
 full-10%              158  0.505413
 full-25%              396  0.533047
 full-50%              792  0.936930
 full-75%             1188  0.946171
full-100%             1584  0.958730 

Part 2 techniques placed on the curve (protocol max_len 128 / batch 8 / 20 epochs):


,technique,rows_used,f1_macro,real_label_equivalent
0,2c zero-shot (LLM),0,0.9779,>full data
1,2d LLM-generated,392,0.8406,698
2,2e optimal-combo,585,0.7997,657
3,2b back-translation,225,0.6355,496
4,2a 32-shot,32,0.6012,463


### 3d - methodology analysis

Part 3 builds a data-scaling curve by training the same `bert-base-uncased` on 1, 10, 25, 50, 75 and 100 percent of the real training set and scoring every point on the shared `test` split.‍ The curve is used here as a yardstick: each Part 2 technique is placed on it at the macro-F1 it reached, and we read off how many real labels the curve needs to match that score.‍ This real-label equivalent is the central comparison the assignment asks for, because it puts the baselines, the augmentation and generation tricks and the zero-shot LLM on a single common scale instead of four separate experiments.‍

The picture that comes out is the following.‍ The two transparent baselines from Part 1 sit at the bottom: the random floor is around 0.30 macro-F1 and the rule-based lexicon reaches 0.735, which the curve only overtakes somewhere between 25 and 50 percent of the real data.‍ The 32-shot BERT (0.601) and the back-translation set (0.635) are each worth roughly 450 to 500 real labels on the curve, even though they only ever saw 32 real annotations; this is the regularisation effect discussed in Part 2b, not new signal.‍ The LLM-generated run is the clear winner among the limited-label methods at 0.841, equivalent to about 700 real labels, because it injects genuinely new examples (in particular the missing negatives) rather than paraphrases of the same 32 sentences.‍ The pooled optimal-combo (0.800, about 660 real labels) lands below it, which is exactly the curation lesson from Part 2e: adding the noisy back-translation rows to the clean generated data dilutes the signal.‍ The zero-shot LLM is off the chart, at 0.978 it is above the curve's maximum of 0.959 at the full 1,584 labels, so on this benchmark no amount of real data fed to BERT matches it.‍

The data-efficiency reading is therefore consistent with the rest of the project.‍ What moves BERT is information, not label count on its own: 32 labels plus generated data beat the 396 real labels of the 25 percent curve point under this comparison, while paraphrasing 32 labels into 225 rows barely moves the score.‍ Real data still wins eventually, the curve climbs steeply once it crosses about 50 percent, but it never closes the gap to the LLM.‍

This comparison has to be read with three caveats.‍

1.‍ Protocol mismatch.‍ The Part 2 techniques were trained with `max_len 128`, batch 8 and 20 epochs, while the Part 3 curve uses a deliberately lighter `max_len 64`, batch 16 and 3 epochs so that the six curve trainings stay affordable on CPU.‍ The two sets of points are therefore not strictly apples-to-apples: part of the height of the Part 2 dots over the nearby curve comes from the heavier protocol, not from the extra data.‍ The real-label equivalents above are accordingly optimistic and should be read as directional, not exact.‍ The clean fix is to re-evaluate the Part 2 techniques under the curve's protocol, or to re-train the curve under the Part 2 protocol, so that every point on the chart shares one training schema.‍

2.‍ Contamination and cost for the LLM row.‍ As argued in Part 2c, Financial PhraseBank is a public 2014 dataset and a frontier model has very likely seen it during pretraining, so the 0.978 ceiling may partly measure memorisation rather than generalisation; it is also a paid, rate-limited, black-box API, set against a curve of deterministic offline BERT runs.‍

3.‍ Low-data instability.‍ The curve is non-monotonic in the minority class at the low end: at 10 and 25 percent the model collapses toward the neutral majority (negative F1 of 0.0) even while accuracy still looks reasonable, and every point is a single-seed estimate.‍ This makes the exact crossing points noisy, precisely in the region below 50 percent where the Part 2 dots sit.‍

With more compute the obvious next steps are to put every method on one training schema and average over several seeds so the real-label equivalents become exact rather than indicative, to add a domain state-of-the-art model such as `ProsusAI/finbert` as a fine-tuned ceiling for the curve, and to test the contamination question for the LLM on an out-of-distribution split that did not exist at the model's training cut-off.‍